# Synergy model interpretation for individual drug combinations

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path for colab/local.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir /  "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

else:
  data_dir = Path("C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir / "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.dge_data import (
    simple_interaction_score,
    eob_score,
    get_all_synergy_data
)

# Bliss score and simple interaction score
data_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = eob_score,
    time_matched = True
)

# Drop genes with NA values
data_df = data_df.dropna(axis = 1)

# Load annotations
annotations = pd.read_table(annot_path, sep = "\t")
annotations.set_index("TIGR4.old", inplace = True, drop = True)

## Feature interpretation

Train and extract features.

In [ ]:
from sklearn.model_selection import KFold
from src.train import run_nested_pls_cv
from src.interpret import cv_feature_importances, plot_top_features

# Set drug combination
combo = "CIP+VNC"

# Random splits
cv = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 111
)

# Isolate cefcip data
df = data_df[data_df["drug_id"] == combo]

# Get splits
splits = list(cv.split(df))

# Get performance
scores = run_nested_pls_cv(
        df = df,
        splits = splits,
        synergy = True
)

# Get feature importances
features = cv_feature_importances(
    df = df,
    splits = splits
)

# Plot top features
plot_top_features(
    coef_df = features,
    annot_df = annotations,
    top_n = 30,
    xlabel = "Coefficient mean over 5 folds",
    title = f"Top 30 features predictive of {combo} synergy"   
)

GSEA on ranked feature list.

In [ ]:
from src.interpret import run_custom_gsea
from gseapy import dotplot, barplot

# Run pre-ranked GSEA
gs = run_custom_gsea(
    coef_df = features,
    annot_df = annotations,
    set_col = "Category1",
    seed = 111
)

fig, ax = plt.subplots()
barplot(
    gs.res2d,
    column = "FDR q-val",
    cutoff = 0.05,
    figsize = (10, 10),
    ax = ax
)
ax.set_title(f"Top enriched pathways from genes most predictive of {combo} synergy", fontsize = "x-large")

Heatmap of top predictive genes' interaction scores.

In [ ]:
# Get dataframe ordered by feature importances
ordered = df[features.index.to_list() + ["synergy_score"]]

plt.figure(figsize = (15, 10))
sns.heatmap(ordered.T[:30], cmap = "coolwarm")
plt.title(f"Heatmap of interaction scores for top 30 genes predictive of {combo} synergy")
plt.tight_layout()

Mann-Whitney U test + FDR on top 100 predictive genes, splitting groups as high and low synergy.

In [ ]:
from scipy.stats import mannwhitneyu, false_discovery_control

# Add column of high and low synergy
ordered["group"] = ordered["synergy_score"].transform(lambda x: "high" if x > 0 else "low")

# Run welch t-test for each 
gene_cols = ordered.columns[ordered.columns.str.contains("SP")][:100]

# Store results
results = []

# Presplit the groups and use only top 100 predictive gnes
high = ordered[ordered["group"] == "high"]
low = ordered[ordered["group"] == "low"]

for gene in gene_cols:
    x = high[gene]
    y = low[gene]

    # Mann whitney 
    u, p = mannwhitneyu(x, y)
    results.append({
        "gene": gene,
        "p_value": p
    })

results = pd.DataFrame(results)

# FDR
results["fdr"] = false_discovery_control(results["p_value"], method = "bh")
results = results.set_index("gene")

# Filter to hits fdr < 0.05
hits = pd.merge(results[results["fdr"] < 0.05], annotations, left_index = True, right_index = True, how = "left").sort_values("fdr")
hits["-log10(p-value)"] = -np.log10(hits["p_value"])

# Plot
fig, ax = plt.subplots(figsize = (7,7))
sns.barplot(hits, y = hits.index, x = "p_value", ax = ax)
ax.set_title(f"Genes significantly different between high and low synergy groups ({combo})")

secax = ax.secondary_yaxis("left")
secax.set_yticks(range(hits.shape[0]))
secax.set_yticklabels(hits["Product"])
secax.spines["left"].set_position(("outward", 130))
secax.set_ylabel("Product")

Stripplot for visualizing individual genes between groups.

In [ ]:
plot = ordered[hits.index.to_list() + ["group"]]

fig, ax = plt.subplots(3, 3, figsize = (15, 15))
ax = ax.flatten()

for i in range(9):
    gene = plot.columns[i]
    sns.stripplot(
        data = plot, 
        x = "group", 
        y = gene,
        hue = "group",
        order = ["low", "high"],
        palette = "seismic",
        ax = ax[i]
    )
    p_val = hits["p_value"].loc[gene]
    text = f"p-value: {p_val:.2e}"
    ax[i].text(
        0.08, 0.08, 
        text, 
        transform = ax[i].transAxes,   
        fontsize = 10, 
        verticalalignment = 'bottom', 
        horizontalalignment = 'left',
    )
    ax[i].set_ylabel(gene)
    ax[i].set_xlabel("Synergy group")
fig.suptitle(
    f"Interaction scores in high and low synergy groups for {combo}",
    fontsize = 14,
    y = 1.02
)
plt.tight_layout()
plt.show()

In [ ]:
cols_of_interest = ["Product", "GENE.CATEGORY", "Tag1", "Product", "Gene.Name", "GO.terms..biological.process."]
top_hits = annotations.loc[plot.columns[:9]][cols_of_interest]